# Results Analysis

Load exported debate JSON files from `results/` and compare score distributions, winners, penalties, and cross-run variance.

In [ ]:
import json
from pathlib import Path

import pandas as pd

results_dir = Path("results")
files = sorted(results_dir.glob("*.json"))
records = []
for f in files:
    data = json.loads(f.read_text(encoding="utf-8"))
    for side in ("pro", "con"):
        score = data.get("final_scores", {}).get(side, {})
        penalties = score.get("penalties_applied", [])
        penalty_total = score.get("penalty_total", 0)
        records.append({
            "file": f.name,
            "topic": data.get("topic", ""),
            "side": side,
            "total": score.get("total", 0),
            "penalty_total": penalty_total,
            "penalty_count": len(penalties),
            "winner": data.get("winner", ""),
            "rounds": len(data.get("rounds", [])),
        })
df = pd.DataFrame(records)
print(f"Loaded {len(files)} debate results ({len(df)} rows)")
df.head()

## Score Comparison Bar Chart

In [ ]:
import matplotlib.pyplot as plt

pivot = df.pivot_table(index="file", columns="side", values="total", aggfunc="first")
pivot.plot(kind="bar", color=["red", "green"], figsize=(12, 5))
plt.title("Pro vs Con Total Scores per Debate")
plt.ylabel("Score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Cross-Run Variance (Line Chart)

In [ ]:
for side, color in [("pro", "green"), ("con", "red")]:
    scores = df[df["side"] == side]["total"].reset_index(drop=True)
    plt.plot(scores.index, scores.values, marker="o", label=side.capitalize(), color=color)
plt.title("Score Variance Across Runs")
plt.xlabel("Run Index")
plt.ylabel("Total Score")
plt.legend()
plt.tight_layout()
plt.show()

## Penalty Heatmap

In [ ]:
penalty_pivot = df.pivot_table(index="file", columns="side", values="penalty_total", aggfunc="first", fill_value=0)
plt.imshow(penalty_pivot.values, cmap="YlOrRd", aspect="auto")
plt.colorbar(label="Penalty Points")
plt.xticks(range(len(penalty_pivot.columns)), penalty_pivot.columns)
plt.yticks(range(len(penalty_pivot.index)), [n[:20] for n in penalty_pivot.index], fontsize=7)
plt.title("Penalty Heatmap per Debate")
plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
print(df.groupby("side")["total"].describe().round(2))
print("\nWinner distribution:")
print(df.drop_duplicates("file")["winner"].value_counts())

In [ ]:
penalty_details = []
for f in files:
    data = json.loads(f.read_text(encoding="utf-8"))
    for rnd in data.get("rounds", []):
        for pen in rnd.get("penalties", []):
            penalty_details.append({
                "file": f.name,
                "round": rnd.get("round_number"),
                "agent": pen.get("agent", ""),
                "type": pen.get("type", ""),
                "points": pen.get("points", 0),
                "reason": pen.get("reason", ""),
            })
if penalty_details:
    pdf = pd.DataFrame(penalty_details)
    print(pdf.groupby(["type", "agent"])["points"].agg(["count", "sum"]))
else:
    print("No penalties recorded.")